# 00 — Setup and a NumPy refresher

Before we implement anything, we need fluency in the one library this repo leans on.
Every algorithm here is written in NumPy, so the vocabulary of **shapes**,
**broadcasting**, and **vectorisation** is the vocabulary of everything that follows.

**What you'll be able to do by the end:**

- Predict the shape of any array operation before running it
- Replace a Python loop with a vectorised expression
- Read `(n, d) @ (d, k) -> (n, k)` and know instantly what it means

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(42)

## Shapes are the whole game

Almost every bug you will hit in this repo is a shape bug. Get in the habit of
printing shapes constantly.

The convention used throughout: **`X` has shape `(n_samples, n_features)`.**
Rows are examples, columns are features. Always.

In [2]:
X = rng.normal(size=(100, 3))   # 100 samples, 3 features
y = rng.normal(size=(100,))     # 100 targets

print("X:", X.shape)
print("y:", y.shape)
print("X[0]:", X[0].shape, "-> one sample")
print("X[:, 0]:", X[:, 0].shape, "-> one feature across all samples")

X: (100, 3)
y: (100,)
X[0]: (3,) -> one sample
X[:, 0]: (100,) -> one feature across all samples


## Broadcasting: the rule that saves you from loops

NumPy lines up shapes **from the right**. Two dimensions are compatible when they
are equal, or when one of them is 1. A missing dimension counts as 1.

That single rule is why we can subtract a per-feature mean from an entire dataset
in one expression.

In [3]:
means = X.mean(axis=0)          # (3,)
centered = X - means            # (100, 3) - (3,) -> broadcasts across rows

print("means:", means.shape, means)
print("centered mean (should be ~0):", centered.mean(axis=0))

# The classic gotcha: (n,) and (n, 1) behave very differently.
a = np.arange(3)
print("\n(3,) - (3,)   ->", (a - a).shape)
print("(3,1) - (3,)  ->", (a.reshape(-1, 1) - a).shape, " <- outer difference, not elementwise!")

means: (3,) [-0.029 -0.013 -0.081]
centered mean (should be ~0): [ 0. -0.  0.]

(3,) - (3,)   -> (3,)
(3,1) - (3,)  -> (3, 3)  <- outer difference, not elementwise!


## Vectorisation: why loops are the enemy

The mathematically identical computation, done two ways. The result is the same;
the runtime is not. This gap is the reason we can train a neural network in a
notebook at all.

In [4]:
import time

A = rng.normal(size=(400, 200))
B = rng.normal(size=(200, 100))

start = time.perf_counter()
slow = np.zeros((400, 100))
for i in range(400):
    for j in range(100):
        slow[i, j] = np.sum(A[i, :] * B[:, j])
loop_time = time.perf_counter() - start

start = time.perf_counter()
fast = A @ B
vec_time = time.perf_counter() - start

print("identical results:", np.allclose(slow, fast))
print(f"loop:       {loop_time * 1000:8.1f} ms")
print(f"vectorised: {vec_time * 1000:8.1f} ms")
print(f"speedup:    {loop_time / vec_time:8.0f}x")

identical results: True
loop:          247.4 ms
vectorised:      1.2 ms
speedup:         204x


## Numerical stability, early

`exp` overflows. `log(0)` is `-inf`. These are not edge cases you will avoid —
they are the default outcome of a naive implementation, and they show up as
`nan` in your loss halfway through training.

The fix for softmax is to subtract the row maximum. This changes nothing
mathematically (the constant cancels in the ratio) but keeps every exponent `<= 0`.

In [5]:
z = np.array([1000.0, 1001.0, 1002.0])

naive = np.exp(z) / np.sum(np.exp(z))       # inf / inf
stable = np.exp(z - z.max()) / np.sum(np.exp(z - z.max()))

print("naive: ", naive, " <- overflowed")
print("stable:", stable, " sums to", stable.sum())

naive:  [nan nan nan]  <- overflowed
stable: [0.09  0.245 0.665]  sums to 0.9999999999999999


/sessions/wonderful-gracious-goldberg/tmp/ipykernel_9/1725025766.py:3: RuntimeWarning: overflow encountered in exp
  naive = np.exp(z) / np.sum(np.exp(z))       # inf / inf
/sessions/wonderful-gracious-goldberg/tmp/ipykernel_9/1725025766.py:3: RuntimeWarning: invalid value encountered in divide
  naive = np.exp(z) / np.sum(np.exp(z))       # inf / inf


## Check your setup

If the cell below runs, you are ready for chapter 01.

In [6]:
from mlscratch.utils import numerical_gradient, relative_error

f = lambda x: float(np.sum(x ** 2))     # gradient is 2x
x = np.array([1.0, 2.0, 3.0])

print("numeric: ", numerical_gradient(f, x.copy()))
print("analytic:", 2 * x)
print("rel error:", relative_error(numerical_gradient(f, x.copy()), 2 * x))
print("\nSetup OK.")

numeric:  [2. 4. 6.]
analytic: [2. 4. 6.]
rel error: 2.548006250370796e-11

Setup OK.


---

**Next:** [01 — Linear regression from scratch](01_linear_regression.ipynb)